In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

In [ ]:

"""
    Input:
        games_path (str)
            Path to league games CSV (e.g., ../csv/us_combined_data.csv)
        standings_path (str)
            Path to end-of-season ranks CSV  (e.g., ../csv/end_of_season/end_of_season_us.csv)
        out_path (str or None, default=None)
            If provided, write the updated games to this CSV.

    Returns:
        pd.DataFrame:
            original games + home_rank, away_rank, is_upset (1 or 0)

    How it works:
        We add the rank of each team to the game, so we know home_rank and away_rank.
        Uses `result` + ranks to set is_upset (1 = upset, 0 = not an upset).
        So if a lower ranked team wins then it is an upset.
        The code then writes this new csv as us_combined_with_upset.csv
"""
def add_upset_column_combined(games_path, standings_path, out_path=None):
    g = pd.read_csv(games_path)
    s = (pd.read_csv(standings_path)[["league", "season", "team_id", "rank"]]
         .dropna(subset=["rank"]))

    g = g.merge(
        s.rename(columns={"team_id": "team1", "rank": "home_rank"}),
        how="left", on=["league", "season", "team1"]
    )
    g = g.merge(
        s.rename(columns={"team_id": "team2", "rank": "away_rank"}),
        how="left", on=["league", "season", "team2"]
    )

    g["is_upset"] = pd.Series(0, index=g.index, dtype="Int64")

    is_tie = g["result"] == 0
    non_tie = ~is_tie
    ranks_present = g["home_rank"].notna() & g["away_rank"].notna()

    stronger_home = g["home_rank"] < g["away_rank"]

    upset_homewin = (g["result"] == 1) & (~stronger_home)
    upset_awaywin = (g["result"] == -1) & (stronger_home)
    upset_mask = non_tie & ranks_present & (upset_homewin | upset_awaywin)

    g.loc[upset_mask, "is_upset"] = 1
    g.loc[non_tie & (~ranks_present), "is_upset"] = pd.NA

    if out_path is not None:
        g.to_csv(out_path, index=False)
        print(f"Wrote games with is_upset to {out_path} (rows: {len(g):,})")

    return g

In [ ]:
# get all the needed paths
games_path = "../../data/us_leagues/csv/game_by_game/pure_skill/us_combined_pure_skill.csv"
standings_path = "../../data/us_leagues/csv/end_of_season/pure_skill/end_of_season_us_pure_skill.csv"
out_combined = "../../output/us_leagues/upset_frequency/pure_skill/csv/us_combined_with_upset_pure_skill.csv"

# build us_combined_with_upset.csv
all_games = add_upset_column_combined(games_path, standings_path, out_path=out_combined)

# keep only valid rows (all rows should be valid)
valid_games = all_games[all_games["is_upset"].notna()].copy()
leagues = sorted(valid_games["league"].unique())

# season level upset frequency by league
season_by_league = (
    valid_games
    .groupby(["league", "season"])["is_upset"]
    .mean()
    .reset_index()
    .rename(columns={"is_upset": "upset_freq"})
)

In [ ]:

"""
Input:
    df_one_league (pd.DataFrame)
        one league's games with is_upset

Returns:
    pd.DataFrame
        per-team upset_rate table sorted descending

How it works:
    Convert games into two rows per game (home and away team views).
    Group by team and take the mean of is_upset.
    This will give us the upset percentage.
"""
def team_upset_rates(df_one_league):
    teams_long = pd.concat([
        df_one_league[["team1", "is_upset"]].rename(columns={"team1": "team"}),
        df_one_league[["team2", "is_upset"]].rename(columns={"team2": "team"}),
    ], ignore_index=True).dropna(subset=["is_upset"])
    return (
        teams_long.groupby("team")["is_upset"].mean()
        .reset_index()
        .rename(columns={"is_upset": "upset_rate"})
        .sort_values("upset_rate", ascending=False)
    )


In [ ]:

"""
    Input:
        league_name (str) (e.g., "NFL", "NBA", "MLB")
        league_df (pd.DataFrame)
            DataFrame for one leagues with an is_upset column
    Returns:
        None
            code display a bar chart of each teams upset rate
    How it works:
        Takes each game and covertes into two rows one for home persepctive and one for away perspective.
        Groups by team and gets the upset percenatge by taking mean of is_upset.
        Then the results are then plotted.
"""

def plot_team_upset_bars(league_name, league_df):
    teams_long = pd.concat([
        league_df[["team1", "is_upset"]].rename(columns={"team1":"team"}),
        league_df[["team2", "is_upset"]].rename(columns={"team2":"team"}),
    ], ignore_index=True).dropna(subset=["is_upset"])

    rates = (teams_long.groupby("team")["is_upset"].mean()
             .reset_index().rename(columns={"is_upset":"upset_rate"})
             .sort_values("upset_rate", ascending=False))

    n = len(rates)
    fig_w = max(12, n * 0.35)

    plt.figure(figsize=(fig_w, 6))
    plt.bar(rates["team"].astype(str), rates["upset_rate"])
    plt.xticks(rotation=45, ha="right")
    ax = plt.gca()
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    plt.ylabel("Upset Frequency (%)")
    plt.title(f"{league_name} Team Upset Frequencies (Pure Skill Simulation)")
    plt.ylim(bottom=0.0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-league bar charts 
for lg in leagues:
    plot_team_upset_bars(lg, valid_games[valid_games["league"] == lg])

In [ ]:

# Box distribution of seasonal upset frequencies by league
box_source = season_by_league.copy()
plt.figure()
data = [grp["upset_freq"].values for _, grp in box_source.groupby("league")]
labels = [lg for lg, _ in box_source.groupby("league")]
plt.boxplot(data, labels=labels)
plt.ylabel("Upset Frequency (%)")
ax = plt.gca()
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
plt.title("Distribution of Upset Frequencies by League (Pure Skill Simulation)")
plt.ylim(bottom=0.0)
plt.tight_layout()
plt.show()


In [ ]:
# Upset Frequency over time by league
line = season_by_league.sort_values(["league", "season"])
fig, ax = plt.subplots()
for lg, g in line.groupby("league"):
    ax.plot(g["season"].astype(int), g["upset_freq"],
            marker="o", markersize=3, linewidth=1.5, label=lg)

ax.set_xlabel("Season")
ax.set_ylabel("Upset Frequency (%)")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
ax.set_title("Upset Frequency Over Time by League (Pure Skill Simulation)")
ax.set_ylim(0.0, 0.5)
ax.set_yticks([i/100 for i in range(0, 51, 5)])

xmin, xmax = int(line["season"].min()), int(line["season"].max())
decades = list(range((xmin//10)*10, (xmax//10)*10 + 1, 10))
ax.set_xticks(decades)
ax.set_xticklabels([str(d) for d in decades])
ax.set_xlim(xmin - 0.5, xmax + 0.5)

ax.legend()
fig.tight_layout()